**Система:** блокчейн записей о собаках  
**Формат записи:** JSON  
**Поля:** кличка, пол, возраст, корм, цена корма  
**API:** Flask  
**Механизм подтверждения:** Proof of Work

На странице со списком записей рассчитывается количество собак для каждого возраста.

## 0. Установка и импорт библиотек

In [ ]:
!pip -q install flask flask-cors

In [ ]:
import json
import time
from collections import Counter
from datetime import datetime
from hashlib import sha256

from flask import Flask, request, jsonify, render_template_string
from flask_cors import CORS
from IPython.display import display, HTML

## 1. Структура JSON

In [ ]:
dog_json_example = {
    "name": "Бобик",
    "gender": "male",
    "age": 3,
    "food": "Royal Canin",
    "food_price": 2500.50,
}

print(
    json.dumps(
        dog_json_example,
        ensure_ascii=False,
        indent=2
    )
)

## 2. Блокчейн

In [ ]:
class Block:
    def __init__(
        self,
        index,
        transactions,
        timestamp,
        previous_hash,
        nonce=0,
    ):
        self.index = index
        self.transactions = transactions
        self.timestamp = timestamp
        self.previous_hash = previous_hash
        self.nonce = nonce

    def compute_hash(self):
        block_string = json.dumps(
            self.__dict__,
            sort_keys=True,
            ensure_ascii=False
        )

        return sha256(
            block_string.encode("utf-8")
        ).hexdigest()


class Blockchain:
    difficulty = 2

    def __init__(self):
        self.unconfirmed_transactions = []
        self.chain = []
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis = Block(
            index=0,
            transactions=[],
            timestamp=time.time(),
            previous_hash="0",
        )

        genesis.hash = genesis.compute_hash()
        self.chain.append(genesis)

    @property
    def last_block(self):
        return self.chain[-1]

    def proof_of_work(self, block):
        block.nonce = 0
        computed_hash = block.compute_hash()

        prefix = "0" * self.difficulty

        while not computed_hash.startswith(prefix):
            block.nonce += 1
            computed_hash = block.compute_hash()

        return computed_hash

    def is_valid_proof(self, block, block_hash):
        return (
            block_hash.startswith(
                "0" * self.difficulty
            )
            and block_hash == block.compute_hash()
        )

    def add_block(self, block, proof):
        if block.previous_hash != self.last_block.hash:
            raise ValueError("Некорректная ссылка на предыдущий блок")

        if not self.is_valid_proof(block, proof):
            raise ValueError("Некорректное доказательство")

        block.hash = proof
        self.chain.append(block)

    def add_new_transaction(self, transaction):
        tx = transaction.copy()
        tx["timestamp"] = time.time()

        self.unconfirmed_transactions.append(tx)

    def mine(self):
        if not self.unconfirmed_transactions:
            return False

        new_block = Block(
            index=self.last_block.index + 1,
            transactions=self.unconfirmed_transactions.copy(),
            timestamp=time.time(),
            previous_hash=self.last_block.hash,
        )

        proof = self.proof_of_work(new_block)
        self.add_block(new_block, proof)

        self.unconfirmed_transactions = []
        return True


blockchain = Blockchain()

## 3. Flask API

In [ ]:
app = Flask(__name__)
CORS(app)


@app.route("/")
def api_index():
    return jsonify({
        "service": "Dog Blockchain API",
        "endpoints": [
            "/new_dog",
            "/dogs",
            "/age_stats",
            "/chain",
            "/mine",
            "/pending_tx",
            "/dashboard",
        ],
    })


@app.route("/new_dog", methods=["POST"])
def new_dog():
    data = request.get_json(force=True)

    required_fields = [
        "name",
        "gender",
        "age",
        "food",
        "food_price",
    ]

    missing = [
        field
        for field in required_fields
        if data.get(field) in (None, "")
    ]

    if missing:
        return jsonify({
            "error": "Отсутствуют поля",
            "fields": missing,
        }), 400

    try:
        data["age"] = int(data["age"])
        data["food_price"] = float(
            data["food_price"]
        )
    except (TypeError, ValueError):
        return jsonify({
            "error": "Некорректный тип данных"
        }), 400

    blockchain.add_new_transaction(data)

    return jsonify({
        "status": "success",
        "pending_count": len(
            blockchain.unconfirmed_transactions
        ),
    }), 201

In [ ]:
def collect_dogs(include_pending=True):
    dogs = []

    for block in blockchain.chain:
        for tx in block.transactions:
            item = tx.copy()
            item["block_index"] = block.index
            item["block_hash"] = block.hash

            if "timestamp" in item:
                item["timestamp_formatted"] = (
                    datetime.fromtimestamp(
                        item["timestamp"]
                    ).strftime("%Y-%m-%d %H:%M:%S")
                )

            dogs.append(item)

    if include_pending:
        for tx in blockchain.unconfirmed_transactions:
            item = tx.copy()
            item["block_index"] = "pending"
            item["block_hash"] = "pending"
            dogs.append(item)

    return dogs


@app.route("/dogs")
def get_dogs():
    dogs = collect_dogs(
        include_pending=True
    )

    return jsonify({
        "dogs": dogs,
        "total": len(dogs),
        "pending": len(
            blockchain.unconfirmed_transactions
        ),
    })


@app.route("/age_stats")
def get_age_stats():
    confirmed_dogs = collect_dogs(
        include_pending=False
    )

    counts = Counter(
        dog["age"]
        for dog in confirmed_dogs
    )

    return jsonify({
        "age_statistics": dict(
            sorted(counts.items())
        ),
        "total_unique_ages": len(counts),
    })


@app.route("/pending_tx")
def get_pending_transactions():
    return jsonify({
        "count": len(
            blockchain.unconfirmed_transactions
        ),
        "pending_transactions":
            blockchain.unconfirmed_transactions,
    })


@app.route("/mine")
def mine():
    result = blockchain.mine()

    if not result:
        return jsonify({
            "status": "error",
            "message": "Нет неподтвержденных записей",
        }), 400

    return jsonify({
        "status": "success",
        "block_index": blockchain.last_block.index,
        "block_hash": blockchain.last_block.hash,
        "transactions_count": len(
            blockchain.last_block.transactions
        ),
    })

In [ ]:
@app.route("/chain")
def get_chain():
    chain_data = []

    for block in blockchain.chain:
        chain_data.append({
            "index": block.index,
            "timestamp": block.timestamp,
            "transactions": block.transactions,
            "transactions_count": len(
                block.transactions
            ),
            "previous_hash": block.previous_hash,
            "hash": block.hash,
            "nonce": block.nonce,
        })

    return jsonify({
        "length": len(chain_data),
        "chain": chain_data,
    })

## 4. Веб-страница со списком собак

In [ ]:
DASHBOARD_TEMPLATE = """
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Список собак</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 1100px;
            margin: 30px auto;
            padding: 0 20px;
        }

        table {
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
        }

        th, td {
            border: 1px solid #bbb;
            padding: 8px;
        }

        th {
            background: #eee;
        }

        .stats {
            display: flex;
            gap: 10px;
            flex-wrap: wrap;
        }

        .stat {
            border: 1px solid #aaa;
            padding: 10px 15px;
        }
    </style>
</head>
<body>
    <h1>Список собак в блокчейне</h1>

    <h2>Количество собак по возрастам</h2>

    <div class="stats">
        {% for age, count in age_stats.items() %}
            <div class="stat">
                Возраст {{ age }}:
                <strong>{{ count }}</strong>
            </div>
        {% endfor %}
    </div>

    <h2>Записи</h2>

    <table>
        <thead>
            <tr>
                <th>Кличка</th>
                <th>Пол</th>
                <th>Возраст</th>
                <th>Корм</th>
                <th>Цена корма</th>
                <th>Блок</th>
            </tr>
        </thead>
        <tbody>
            {% for dog in dogs %}
                <tr>
                    <td>{{ dog.name }}</td>
                    <td>{{ dog.gender }}</td>
                    <td>{{ dog.age }}</td>
                    <td>{{ dog.food }}</td>
                    <td>{{ "%.2f"|format(dog.food_price) }}</td>
                    <td>{{ dog.block_index }}</td>
                </tr>
            {% endfor %}
        </tbody>
    </table>
</body>
</html>
"""


@app.route("/dashboard")
def dashboard():
    dogs = collect_dogs(
        include_pending=True
    )

    confirmed = collect_dogs(
        include_pending=False
    )

    counts = Counter(
        dog["age"]
        for dog in confirmed
    )

    return render_template_string(
        DASHBOARD_TEMPLATE,
        dogs=dogs,
        age_stats=dict(sorted(counts.items())),
    )

## 5. Заполнение блокчейна

In [ ]:
sample_dogs = [
    {
        "name": "Бобик",
        "gender": "male",
        "age": 3,
        "food": "Royal Canin",
        "food_price": 2500.50,
    },
    {
        "name": "Шарик",
        "gender": "male",
        "age": 5,
        "food": "Purina Pro Plan",
        "food_price": 3200.00,
    },
    {
        "name": "Люси",
        "gender": "female",
        "age": 2,
        "food": "Acana",
        "food_price": 4100.75,
    },
    {
        "name": "Рекс",
        "gender": "male",
        "age": 4,
        "food": "Pedigree",
        "food_price": 1800.00,
    },
    {
        "name": "Белла",
        "gender": "female",
        "age": 2,
        "food": "Hills Science Diet",
        "food_price": 3600.25,
    },
]

client = app.test_client()

for dog in sample_dogs:
    response = client.post(
        "/new_dog",
        json=dog
    )
    print(
        dog["name"],
        response.status_code,
        response.get_json()
    )

## 6. Майнинг

In [ ]:
print("До майнинга:")
print(client.get("/pending_tx").get_json())

mine_response = client.get("/mine")

print("\nМайнинг:")
print(mine_response.get_json())

print("\nПосле майнинга:")
print(client.get("/pending_tx").get_json())

## 7. Статистика по возрастам

In [ ]:
age_stats = client.get(
    "/age_stats"
).get_json()

print(
    json.dumps(
        age_stats,
        ensure_ascii=False,
        indent=2
    )
)

## 8. Цепочка блоков

In [ ]:
chain_info = client.get(
    "/chain"
).get_json()

print("Количество блоков:", chain_info["length"])

for block in chain_info["chain"]:
    print(
        f"Блок {block['index']}: "
        f"{block['transactions_count']} записей, "
        f"nonce={block['nonce']}, "
        f"hash={block['hash'][:18]}..."
    )

## 9. Предпросмотр страницы

In [ ]:
html = client.get(
    "/dashboard"
).data.decode("utf-8")

display(HTML(html))

## Итог

In [ ]:
dogs_info = client.get(
    "/dogs"
).get_json()

stats_info = client.get(
    "/age_stats"
).get_json()

print("Всего записей:", dogs_info["total"])
print("Возрастная статистика:")
print(stats_info["age_statistics"])
print("Блоков:", client.get("/chain").get_json()["length"])